## Package Imports & Common Variables

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import datetime
import plotly.graph_objects as go


load_dotenv()
ticker = ''
path_stockdata = os.path.join(os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList'), ticker)
path_analysis_csv = os.path.join(path_stockdata, f'{ticker}--Analysis_Ownership-s1v1.csv')

today = datetime.date.today()
cutoff = today.year - 20
cutoff10 = today.year - 10
cutoff5 = today.year - 5
cutoff3 = today.year - 3


## Data Imports & Cleaning

In [ ]:
%%capture
df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Annual_10K-s1v1.csv'), index_col=0)
df0 = df0.dropna(subset=['FiscalYear'])
df0[['FiscalYear', 'FiscalMonth']] = df0[['FiscalYear', 'FiscalMonth']].astype(int)

In [ ]:
df0

In [ ]:
%%capture

owner_df = df0[['FiscalYear', 'Revenue', 'OpCash', 'FreeCash','DivCash', 'StockIssue', 'StockBuyBack',
                'C&E', 'TreasuryStock', 'SharesOutstandingBasic', 'SharesOutstandingDiluted']]
owner_df['OwnerStock'] = (owner_df['StockIssue'] + owner_df['StockBuyBack']).abs()
owner_df['OwnerDiv'] = owner_df['DivCash'].abs()
owner_df['OwnerTot'] = (owner_df['OwnerStock'] + owner_df['OwnerDiv'])
owner_df['OwnerYield'] = round( owner_df['OwnerTot'] / owner_df['Revenue'] * 100 ,2)

In [ ]:
owner_df.tail(10)

## Total Owner Return

In [ ]:
owner_return10 = owner_df['OwnerTot'].tail(10).sum()
owner_div10 = owner_df['OwnerDiv'].tail(10).sum()
owner_stock10 = owner_df['OwnerStock'].tail(10).sum()
rev10 = owner_df['Revenue'].tail(10).sum()
oc10 = owner_df['OpCash'].tail(10).sum()
fc10 = owner_df['FreeCash'].tail(10).sum()

print(f'10 Year total cash outflow to owners: $ {owner_return10}')
print(f'10 Year total dividend cash outflow to owners: $ {owner_div10}')
print(f'10 Year total cash via stock repurchase outflow to owners: $ {owner_stock10}')
print()
print(f'Percent of revenue returned to owners: {round(owner_return10 / rev10 * 100, 2)} %')
print(f'Percent of operating cash returned to owners: {round(owner_return10 / oc10 * 100, 2)} %')
print(f'Percent of free cash returned to owners: {round(owner_return10 / fc10 * 100, 2)} %')

In [ ]:
owner_fig0 = go.Figure(data=[
    go.Bar(name='OpCash', x=owner_df['FiscalYear'].tail(10), y=owner_df['OpCash'].tail(10), offsetgroup=1, marker_color='blue'),
    go.Bar(name='FreeCash', x=owner_df['FiscalYear'].tail(10), y=owner_df['FreeCash'].tail(10), offsetgroup=2, marker_color='green'),
    go.Bar(name='OwnerDiv', x=owner_df['FiscalYear'].tail(10), y=owner_df['OwnerDiv'].tail(10), offsetgroup=3, marker_color='orange'),
    go.Bar(name='OwnerStock', x=owner_df['FiscalYear'].tail(10), y=owner_df['OwnerStock'].tail(10), offsetgroup=3,
           base=owner_df['OwnerDiv'].tail(10), marker_color='yellow')
])
owner_fig0.update_layout(barmode='group')
owner_fig0.update_xaxes(dtick=1)
owner_fig0.show()

In [ ]:
owner_fig1 = go.Figure(data=[
    go.Bar(name='OwnerYield', x=owner_df['FiscalYear'].tail(10), y=owner_df['OwnerYield'].tail(10), offsetgroup=1, marker_color='blue'),
])
owner_fig1.update_layout(barmode='group')
owner_fig1.update_layout(yaxis_title='Yield %', xaxis_title='FiscalYear', title='10 Year Owner Yield', template='plotly_dark')
owner_fig1.update_xaxes(dtick=1)
owner_fig1.show()

## Shares Out

In [ ]:
share_fig1 = go.Figure(data=[
    go.Bar(name='BasicShares', x=owner_df['FiscalYear'].tail(10), y=owner_df['SharesOutstandingBasic'].tail(10), offsetgroup=1, marker_color='blue'),
    go.Bar(name='DilutedShares', x=owner_df['FiscalYear'].tail(10), y=owner_df['SharesOutstandingDiluted'].tail(10), offsetgroup=2, marker_color='green')
])
share_fig1.update_layout(barmode='group')
share_fig1.update_layout(yaxis_title='Count', xaxis_title='FiscalYear', title='10 Year Share Count', template='plotly_dark')
share_fig1.update_xaxes(dtick=1)
share_fig1.show()

In [ ]:
%%capture

share_df = owner_df[['FiscalYear', 'SharesOutstandingDiluted', 'SharesOutstandingBasic']].tail(10)
share_df['DilutedChg'] = share_df['SharesOutstandingDiluted'].pct_change()
share_df['BasicChg'] = share_df['SharesOutstandingBasic'].pct_change()

share_df

In [ ]:
share_fig2 = go.Figure(data=[
   go.Bar(name='BasicChg', x=share_df['FiscalYear'], y=share_df['BasicChg'] * 100, offsetgroup=1, marker_color='blue'),
    go.Bar(name='DilutedChg', x=share_df['FiscalYear'], y=share_df['DilutedChg'] * 100, offsetgroup=2, marker_color='green')
])
share_fig2.update_xaxes(dtick=1)
share_fig2.update_layout(yaxis_title='% Change', xaxis_title='FiscalYear', title='10 Year Share Change', template='plotly_dark')
share_fig2.show()

## Dividend Returns

In [ ]:
%%capture

div_df = owner_df[['FiscalYear', 'Revenue', 'OpCash', 'FreeCash', 'DivCash']]
div_df['DivGro'] = round(div_df['DivCash'].pct_change() * 100, 2)
div_df['DivMargin'] = round(div_df['DivCash'] / div_df['Revenue'] * 100, 2).abs()
div_df['OpCashCover'] = round(div_df['DivCash'] / div_df['OpCash'] * 100, 2).abs()
div_df['FreeCashCover'] = round(div_df['DivCash'] / div_df['FreeCash'] * 100, 2).abs()


In [ ]:
div_df.tail(10)

In [ ]:
divgro10mn = round(div_df['DivGro'].tail(10).mean(), 2)
divgro10md = round(div_df['DivGro'].tail(10).median(), 2)

divgro5mn = round(div_df['DivGro'].tail(5).mean(), 2)
divgro5md = round(div_df['DivGro'].tail(5).median(), 2)

divgro3mn = round(div_df['DivGro'].tail(3).mean(), 2)
divgro3md = round(div_df['DivGro'].tail(3).median(), 2)

print(f'The 10 Year Mean Dividend Growth is: {divgro10mn}%')
print(f'The 10 Year Median Dividend Growth is: {divgro10md}%')
print()
print(f'The 5 Year Mean Dividend Growth is: {divgro5mn}%')
print(f'The 5 Year Median Dividend Growth is: {divgro5md}%')
print()
print(f'The 3 Year Mean Dividend Growth is: {divgro3mn}%')
print(f'The 3 Year Median Dividend Growth is: {divgro3md}%')
print()

In [ ]:
div_fig1 = go.Figure(data=[
   go.Bar(name='DivGro', x=div_df['FiscalYear'].tail(10), y=div_df['DivGro'].tail(10), offsetgroup=1, marker_color='blue')
    ])
div_fig1.update_xaxes(dtick=1)
div_fig1.update_layout(yaxis_title='% Change', xaxis_title='FiscalYear', title='10 Year DivGro', template='plotly_dark')
div_fig1.show()

In [ ]:
div_fig2 = go.Figure(data=[
    go.Bar(name='OpCash', x=div_df['FiscalYear'].tail(10), y=div_df['OpCashCover'].tail(10), offsetgroup=1, marker_color='blue'),
    go.Bar(name='FreeCash', x=div_df['FiscalYear'].tail(10), y=div_df['FreeCashCover'].tail(10), offsetgroup=2, marker_color='Green')
    ])
div_fig2.update_xaxes(dtick=1)
div_fig2.update_layout(yaxis_title='Coverage %', xaxis_title='FiscalYear', title='10 Year Div Coverage', template='plotly_dark')
div_fig2.show()

## Analysis Output

In [ ]:
metrics_json = {
    "type": "ownership",
    "date": today.strftime('%Y-%m-%d'),
    "last_df_date": df0['FiscalYear'].iloc[-1],
    "10year_cash_outflow": owner_return10,
    "10year_div_outflow": owner_div10,
    "10year_stock_outflow": owner_stock10,
    "10year_owner_yield": round(owner_return10 / rev10, 4),
    "10year_divgro_mean": divgro10mn / 100,
    "10year_divgro_median": divgro10md,
    "5year_divgro_mean": divgro5mn / 100,
    "5year_divgro_median": divgro5md,
    "3year_divgro_mean": divgro3mn / 100,
    "3year_divgro_median": divgro3md / 100,
    "divgro_trend": "slowing",
    "div_risk": "low"
}

metrics_json

In [ ]:
metrics_df = pd.DataFrame([metrics_json])
metrics_df

In [ ]:
if os.path.isfile(path_analysis_csv):
    metrics_df.to_csv(path_analysis_csv, mode='a', header=False, index=False)
else:
    metrics_df.to_csv(path_analysis_csv, mode='w', header=True, index=False)

## End Notebook